<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# Load data and preprocess

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import numpy as np
import plotly.graph_objects as go
grid = np.load('../Data/building_case_0.npy')
wind = np.load('../Data/mean_case_0.npy')[:, :, :, [2,1,0]]

FileNotFoundError: [Errno 2] No such file or directory: '../Data/building_case_0.npy'

In [ ]:
# Swap trục Y (axis=1) và Z (axis=2)
grid = np.swapaxes(grid, 1, 2)
# Swap không gian Y <-> Z
wind = np.swapaxes(wind, 1, 2)

# Swap luôn thành phần vector bên trong: (u, v, w) -> (u, w, v)
wind = wind[..., [0, 2, 1]]

print("grid shape sau khi swap:", grid.shape)
print("wind shape sau khi swap:", wind.shape)

In [ ]:
# Quy đổi voxel -> mét (theo ghi chú: 40 voxel ~ 120m, 40 voxel ~ 12m)
# Nếu bạn có thông số chính xác khác, chỉ cần sửa tuple này.
voxel_size_m = np.array([4, 4, 1.5], dtype=float)  # (dx, dy, dz) theo trục (X, Y, Z)# Kích thước thực toàn miền từ grid ban đầu
grid_size_voxel = np.array(grid.shape, dtype=int)
grid_size_m = grid_size_voxel * voxel_size_m

print(f"Grid shape (voxel): {tuple(grid_size_voxel)}")
print(f"Grid physical size (m): X={grid_size_m[0]:.2f}, Y={grid_size_m[1]:.2f}, Z={grid_size_m[2]:.2f}")


In [ ]:
# Vẽ không gian thực (đơn vị mét) bằng Plotly
# Dùng các biến đã có: grid, voxel_size_m, grid_size_m, np


# 1) Lấy tọa độ vật cản (voxel True) và đổi sang mét
obs_idx = np.argwhere(grid)  # (N, 3) theo (x, y, z)
max_points = 120_000  # giới hạn để plot mượt hơn

if len(obs_idx) > max_points:
    step = int(np.ceil(len(obs_idx) / max_points))
    obs_idx = obs_idx[::step]

# đặt điểm ở tâm voxel
obs_m = (obs_idx + 0.5) * voxel_size_m  # (x, y, z) mét

# 2) Tạo wireframe biên toàn miền (m)
X, Y, Z = grid_size_m
corners = np.array([
    [0, 0, 0], [X, 0, 0], [X, Y, 0], [0, Y, 0],
    [0, 0, Z], [X, 0, Z], [X, Y, Z], [0, Y, Z]
], dtype=float)

edges = [
    (0,1),(1,2),(2,3),(3,0),
    (4,5),(5,6),(6,7),(7,4),
    (0,4),(1,5),(2,6),(3,7)
]

bx, by, bz = [], [], []
for a, b in edges:
    bx += [corners[a, 0], corners[b, 0], None]
    by += [corners[a, 1], corners[b, 1], None]
    bz += [corners[a, 2], corners[b, 2], None]

# 3) Plot
fig_real = go.Figure()

fig_real.add_trace(go.Scatter3d(
    x=bx, y=by, z=bz,
    mode="lines",
    line=dict(color="deepskyblue", width=4),
    name="Biên không gian (m)"
))

if len(obs_m) > 0:
    fig_real.add_trace(go.Scatter3d(
        x=obs_m[:, 0], y=obs_m[:, 1], z=obs_m[:, 2],
        mode="markers",
        marker=dict(size=2, color="crimson", opacity=0.35),
        name=f"Vật cản ({len(obs_m):,} điểm)"
    ))

fig_real.update_layout(
    title="Không gian thực (m)",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1200,
    height=650
)

fig_real.show()

# predefine

In [ ]:
min_cubic=[60, 60, 40] # Minimum box size in voxels (120m x 120m x 12m)
# wind_threshold=0.5

In [ ]:
def calculate_global_threshold(wind_data, percentage=0.5):
    """
    Tính toán ngưỡng gió dựa trên phần trăm độ lớn trung bình của toàn bộ không gian.
    
    wind_data: Mảng 4D (dx, dy, dz, 3) chứa các thành phần u, v, w.
    percentage: Tỷ lệ phần trăm mong muốn (mặc định 0.2 tương đương 20%).
    """
    # 1. Tính độ lớn (Magnitude) của vector gió tại mỗi điểm (voxel)
    # Công thức: sqrt(u^2 + v^2 + w^2)
    magnitudes = np.linalg.norm(wind_data, axis=-1)
    
    # 2. Tính giá trị độ lớn trung bình của toàn bộ không gian
    global_avg_magnitude = np.mean(magnitudes)
    
    # 3. Tính ngưỡng theo tỷ lệ phần trăm
    threshold = global_avg_magnitude * percentage
    
    print(f"--- Báo cáo thông số gió toàn không gian ---")
    print(f"Độ lớn gió trung bình (Global Mean): {global_avg_magnitude:.4f} m/s")
    print(f"Ngưỡng đề xuất ({percentage*100}%): {threshold:.4f} m/s")
    
    return threshold

wind_threshold=calculate_global_threshold(wind)

In [ ]:
# v_levels = [0]  # 0 for startpoint, 1 for endpoint, 2 for intermediate point
# --- Sample parameter configuration ---
drone_params = {
    'rho': 1.18,      # kg/m3
    'Cd': 1,
    'Af': 0.25,        # m2
    'm': 4.0  ,          # kg
    'g_acc': np.array([0, 0, -9.81]),
    'A': 0.45,          # m2
    'Pp_hover':65.0  # Watts
}
speed_map = {0:10.0, 1:10.0}
dt = 4
dt_takeoff = 4
dt_landing = 4
Tmax = 60

# Adaptive 3D space discreatation

In [ ]:
import time

class AdaptiveBox:
    def __init__(self, x_rng, y_rng, z_rng, is_obs, avg_w, w_std):
        self.x_rng, self.y_rng, self.z_rng = x_rng, y_rng, z_rng
        self.is_obstacle = is_obs
        self.avg_wind = avg_w
        self.wind_std = w_std
        
    def __repr__(self):
        type_str = "OBSTACLE" if self.is_obstacle else "FREE"
        size = (
            self.x_rng[1] - self.x_rng[0],
            self.y_rng[1] - self.y_rng[0],
            self.z_rng[1] - self.z_rng[0]
        )
        return (
                f"{type_str}\n"
                f"  x_range: {self.x_rng}\n"
                f"  y_range: {self.y_rng}\n"
                f"  z_range: {self.z_rng}\n"
                f"  size: {size}\n"
                f"  avg_wind: {self.avg_wind}\n"
                f"  wind_std: {self.wind_std}"
            )

def partition_space(
    obs_data,
    wind_data,
    x_rng,
    y_rng,
    z_rng,
    wind_threshold=0.5,
    min_cubic=(5, 5, 5),
    voxel_size=[1, 1, 1],
    min_cubic_unit="voxel"  # "voxel" hoặc "meter"
):
    x1, x2 = x_rng
    y1, y2 = y_rng
    z1, z2 = z_rng


    # Trích xuất vùng dữ liệu
    obs_reg = obs_data[x1:x2, y1:y2, z1:z2]
    wind_reg = wind_data[x1:x2, y1:y2, z1:z2, :]

    dx, dy, dz = x2 - x1, y2 - y1, z2 - z1
    size_voxel = np.array([dx, dy, dz], dtype=float)
    size_meter = size_voxel * voxel_size

    # 1) Trạng thái vật cản (hỗ trợ cả bool và 0/1)
    has_obs = np.any(obs_reg)
    all_obs = np.all(obs_reg)

    # 2) Độ biến động gió
    vector_dispersion = 0.0
    avg_w = np.zeros(3, dtype=float)

    # Chỉ xử lý các voxel KHÔNG phải vật cản
    if wind_reg.size > 0:
        free_mask = ~obs_reg.astype(bool)  # True tại voxel tự do
        if np.any(free_mask):
            free_wind = wind_reg[free_mask]   # shape: (N_free, 3)
            avg_w = np.mean(free_wind, axis=0)

            diff = free_wind - avg_w
            sq_norms = np.sum(diff**2, axis=-1)
            vector_dispersion = float(np.sqrt(np.mean(sq_norms)))

    # 3) Điều kiện dừng theo min_cubic
    min_cubic = np.asarray(min_cubic, dtype=float)
    if min_cubic_unit == "meter":
        is_min_size = np.all(size_meter <= min_cubic)
    else:
        is_min_size = np.all(size_voxel <= min_cubic)

    is_stable = (not has_obs) and (vector_dispersion < wind_threshold)

    if all_obs or is_stable or is_min_size:
        return [AdaptiveBox(x_rng, y_rng, z_rng, has_obs, avg_w, vector_dispersion)]

    # 4) Chia theo trục dài nhất theo KÍCH THƯỚC THỰC (m)
    axis = int(np.argmax(size_meter))
    ranges = [x_rng, y_rng, z_rng]
    mid = ranges[axis][0] + (ranges[axis][1] - ranges[axis][0]) // 2

    # tránh chia rỗng gây đệ quy vô hạn
    if mid == ranges[axis][0] or mid == ranges[axis][1]:
        return [AdaptiveBox(x_rng, y_rng, z_rng, has_obs, avg_w, vector_dispersion)]

    r1, r2 = list(ranges), list(ranges)
    r1[axis] = (ranges[axis][0], mid)
    r2[axis] = (mid, ranges[axis][1])

    return (
        partition_space(
            obs_data, wind_data, *r1,
            wind_threshold=wind_threshold,
            min_cubic=min_cubic,
            voxel_size=voxel_size,
            min_cubic_unit=min_cubic_unit
        )
        + partition_space(
            obs_data, wind_data, *r2,
            wind_threshold=wind_threshold,
            min_cubic=min_cubic,
            voxel_size=voxel_size,
            min_cubic_unit=min_cubic_unit
        )
    )

In [ ]:
# Space-size configuration
SHAPE = grid.shape
print(f"Initializing space {SHAPE}...")



# Run partitioning
start = time.time()
# You can tune wind_threshold (e.g., 0.8 for fewer boxes, 0.3 for more detail)
results = partition_space(grid, wind, (0, SHAPE[0]), (0, SHAPE[1]), (0, SHAPE[2]), wind_threshold=wind_threshold, min_cubic=min_cubic)
print(f"Done! Elapsed time: {time.time() - start:.2f}s")
print(f"Total generated boxes: {len(results)}")

In [ ]:
# results=results[:1]  # Giới hạn số box để plot mượt hơn

In [ ]:
results

In [ ]:
# Replot full space in real-world size (meters):
# free cubic, obstacle cubic, wind vectors (free only), obstacle voxels
import plotly.graph_objects as go


voxel_coords_plot = np.empty((0, 3), dtype=float)


def idx_to_m(xyz):
    # xyz: (..., 3) in grid-index units -> meters
    return xyz * voxel_size_m


def wireframe_adaptive(boxes):
    xs, ys, zs = [], [], []
    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7)
    ]
    for b in boxes:
        x1, x2 = b.x_rng
        y1, y2 = b.y_rng
        z1, z2 = b.z_rng

        v_idx = np.array([
            [x1, y1, z1], [x2, y1, z1], [x2, y2, z1], [x1, y2, z1],
            [x1, y1, z2], [x2, y1, z2], [x2, y2, z2], [x1, y2, z2]
        ], dtype=float)

        v = idx_to_m(v_idx)  # convert box corners to meters

        for a, c in edges:
            xs += [v[a, 0], v[c, 0], None]
            ys += [v[a, 1], v[c, 1], None]
            zs += [v[a, 2], v[c, 2], None]
    return xs, ys, zs


boxes_obs = [b for b in results if b.is_obstacle]
boxes_free = [b for b in results if not b.is_obstacle]

x_free, y_free, z_free = wireframe_adaptive(boxes_free)
x_obs, y_obs, z_obs = wireframe_adaptive(boxes_obs)

# Wind vectors (only free boxes)
centers_idx = np.array([
    ((b.x_rng[0] + b.x_rng[1]) / 2, (b.y_rng[0] + b.y_rng[1]) / 2, (b.z_rng[0] + b.z_rng[1]) / 2)
    for b in results
], dtype=float)
centers = idx_to_m(centers_idx)  # meters

vec = np.array([b.avg_wind for b in results], dtype=float)  # assumed m/s direction vector
mag = np.linalg.norm(vec, axis=1)
mask_free = np.array([not b.is_obstacle for b in results], dtype=bool)
valid = (mag > 1e-9) & mask_free

wind_arrow_scale = 3.0  # seconds-like factor: arrow length = speed * scale (meters)
p0 = centers[valid]
p1 = p0 + vec[valid] * wind_arrow_scale

fig_all = go.Figure()

fig_all.add_trace(go.Scatter3d(
    x=x_free, y=y_free, z=z_free,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"Free cubic ({len(boxes_free):,})"
))

# Bật nếu muốn hiện obstacle wireframes
fig_all.add_trace(go.Scatter3d(
    x=x_obs, y=y_obs, z=z_obs,
    mode="lines",
    line=dict(color="orange", width=2),
    name=f"Obstacle cubic ({len(boxes_obs):,})"
))

if len(p0) > 0:
    d = p1 - p0
    fig_all.add_trace(go.Cone(
        x=p1[:, 0], y=p1[:, 1], z=p1[:, 2],
        u=d[:, 0], v=d[:, 1], w=d[:, 2],
        anchor="tip",
        sizemode="absolute",
        sizeref=10.0,
        colorscale=[[0, "lime"], [1, "lime"]],
        showscale=False,
        name="Wind direction (free only)"
    ))

if len(voxel_coords_plot) > 0:
    vox_plot = voxel_coords_plot.copy()
    # nếu voxel_coords_plot đang ở index-space thì đổi sang mét
    if np.nanmax(vox_plot[:, 0]) <= np.max(obs_idx[:, 0]) + 1:
        vox_plot = idx_to_m(vox_plot)

    fig_all.add_trace(go.Scatter3d(
        x=vox_plot[:, 0], y=vox_plot[:, 1], z=vox_plot[:, 2],
        mode="markers",
        marker=dict(size=2, color="crimson", opacity=0.25),
        name=f"Obstacle voxels ({len(vox_plot):,})"
    ))

fig_all.update_layout(
    title="Space (real scale, meters): free/obstacle cubic + wind vectors",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    autosize=False,
    width=1000,
    height=600
)

fig_all.show()


In [ ]:
# Print wind vectors after partitioning
vec_after_partition = np.array([b.avg_wind for b in results], dtype=float)
print(vec_after_partition.shape)

In [ ]:
vec_after_partition[:5]

# Edge generate

In [ ]:
results_free = [b for b in results if not b.is_obstacle]
print(f"Số box còn lại (FREE): {len(results_free)}")

# Vẽ lại chỉ với FREE boxes + vector gió (đã swap Y<->Z theo wireframe_adaptive)
x_free, y_free, z_free = wireframe_adaptive(results_free)

if len(results_free) > 0:
    centers = np.array([
        ((b.x_rng[0] + b.x_rng[1]) / 2,
         (b.y_rng[0] + b.y_rng[1]) / 2,
         (b.z_rng[0] + b.z_rng[1]) / 2)
        for b in results_free
    ], dtype=float)

    vec = np.array([b.avg_wind for b in results_free], dtype=float)
    mag = np.linalg.norm(vec, axis=1)
    valid = mag > 1e-9

    p0 = centers[valid]
    p1 = p0 + vec[valid] * wind_arrow_scale
    d = p1 - p0
else:
    centers = np.empty((0, 3), dtype=float)
    p0 = p1 = d = np.empty((0, 3), dtype=float)

fig_free = go.Figure()

fig_free.add_trace(go.Scatter3d(
    x=x_free, y=y_free, z=z_free,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"Free cubic ({len(results_free):,})"
))

if len(p0) > 0:
    fig_free.add_trace(go.Cone(
        x=p1[:, 0],
        y=p1[:, 2],  # swap Y<->Z
        z=p1[:, 1],  # swap Y<->Z
        u=d[:, 0],
        v=d[:, 2],   # swap Y<->Z
        w=d[:, 1],   # swap Y<->Z
        anchor="tip",
        sizemode="absolute",
        sizeref=10.0,
        colorscale=[[0, "lime"], [1, "lime"]],
        showscale=False,
        name="Wind direction"
    ))

fig_free.update_layout(
    title="Free space only (obstacles removed from results, Y<->Z)",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Z",  # swapped
        zaxis_title="Y",  # swapped
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    autosize=False,
    width=1000,
    height=600
)

fig_free.show()

## Adjacent to each other on the same face

In [ ]:
# Kiểm tra các box tiếp xúc mặt và tạo edge nối tâm 2 box
# Lấy từ results (không dùng wind_boxes_plot)

def to_box_dict(box):
    # Chuẩn hóa về dict {'pos','size',...} để dùng chung cho các cell dưới
    if isinstance(box, dict):
        return box
    return {
        "pos": (box.x_rng[0], box.y_rng[0], box.z_rng[0]),
        "size": (box.x_rng[1] - box.x_rng[0], box.y_rng[1] - box.y_rng[0], box.z_rng[1] - box.z_rng[0]),
        "avg_wind": box.avg_wind,
        "turbulence_level": box.wind_std,
        "pos_m": (box.x_rng[0]  * voxel_size_m[0],
                  box.y_rng[0]  * voxel_size_m[1],
                  box.z_rng[0]  * voxel_size_m[2]),
        "size_m": ((box.x_rng[1] - box.x_rng[0])* voxel_size_m[0],
                   (box.y_rng[1] - box.y_rng[0]) * voxel_size_m[1],
                   (box.z_rng[1] - box.z_rng[0]) * voxel_size_m[2])
    }

source_boxes = [to_box_dict(b) for b in results]

def get_bounds_and_center(box):
    x1, y1, z1 = box["pos"]
    dx, dy, dz = box["size"]
    x2, y2, z2 = x1 + dx, y1 + dy, z1 + dz
    c = ((x1 + x2) / 2, (y1 + y2) / 2, (z1 + z2) / 2)
    return (x1, x2, y1, y2, z1, z2), c

def overlap_positive(a1, a2, b1, b2, eps=1e-9):
    return min(a2, b2) - max(a1, b1) > eps

def face_touch(b1, b2, eps=1e-9):
    x1a, x2a, y1a, y2a, z1a, z2a = b1
    x1b, x2b, y1b, y2b, z1b, z2b = b2

    touch_x = (abs(x2a - x1b) <= eps or abs(x2b - x1a) <= eps) and \
              overlap_positive(y1a, y2a, y1b, y2b, eps) and \
              overlap_positive(z1a, z2a, z1b, z2b, eps)

    touch_y = (abs(y2a - y1b) <= eps or abs(y2b - y1a) <= eps) and \
              overlap_positive(x1a, x2a, x1b, x2b, eps) and \
              overlap_positive(z1a, z2a, z1b, z2b, eps)

    touch_z = (abs(z2a - z1b) <= eps or abs(z2b - z1a) <= eps) and \
              overlap_positive(x1a, x2a, x1b, x2b, eps) and \
              overlap_positive(y1a, y2a, y1b, y2b, eps)

    return touch_x or touch_y or touch_z

# Tiền xử lý bounds + centers
bounds = []
centers = []
for b in source_boxes:
    bd, c = get_bounds_and_center(b)
    bounds.append(bd)
    centers.append(c)
centers = np.array(centers, dtype=float)

# Tìm cặp box tiếp xúc mặt
n = len(source_boxes)
adjacency = {i: set() for i in range(n)}
edge_pairs = []

for i in range(n):
    for j in range(i + 1, n):
        if face_touch(bounds[i], bounds[j]):
            adjacency[i].add(j)
            adjacency[j].add(i)
            edge_pairs.append((i, j))

adjacency = {k: sorted(v) for k, v in adjacency.items()}

print(f"Tổng số box: {n}")
print(f"Số cặp tiếp xúc mặt: {len(edge_pairs)}")
print("5 box đầu tiên và các box tiếp xúc mặt với nó:")
for i in range(min(5, n)):
    print(f"Box {i} -> {adjacency[i]}")

# Tạo mảng edge để vẽ (nối tâm 2 box)
edge_x, edge_y, edge_z = [], [], []
for i, j in edge_pairs:
    c1, c2 = centers[i], centers[j]
    edge_x += [c1[0], c2[0], None]
    edge_y += [c1[1], c2[1], None]
    edge_z += [c1[2], c2[2], None]


In [ ]:
# Vẽ từ kết quả đã lọc (FREE only)
source_boxes_free = [to_box_dict(b) for b in results_free] if "results_free" in globals() else [
    to_box_dict(b) for b in results if not b.is_obstacle
]

# Bounds + centers (index-space)
bounds_free = []
centers_free = []
for b in source_boxes_free:
    bd, c = get_bounds_and_center(b)
    bounds_free.append(bd)
    centers_free.append(c)

centers_free = np.array(centers_free, dtype=float) if len(centers_free) else np.empty((0, 3), dtype=float)

# Face-adjacency trên tập FREE
face_pairs_free = []
for i in range(len(source_boxes_free)):
    for j in range(i + 1, len(source_boxes_free)):
        if face_touch(bounds_free[i], bounds_free[j]):
            face_pairs_free.append((i, j))

# Đổi index -> mét
scale = np.asarray(voxel_size_m, dtype=float)  # (dx, dy, dz) m/voxel

# Line nối tâm các cặp face-adjacent (meter-space)
face_x_free, face_y_free, face_z_free = [], [], []
for i, j in face_pairs_free:
    c1 = centers_free[i] * scale
    c2 = centers_free[j] * scale
    face_x_free += [c1[0], c2[0], None]
    face_y_free += [c1[1], c2[1], None]
    face_z_free += [c1[2], c2[2], None]

# Wireframe FREE boxes (meter-space)
all_x_box, all_y_box, all_z_box = [], [], []
edges = [
    (0,1),(1,2),(2,3),(3,0),
    (4,5),(5,6),(6,7),(7,4),
    (0,4),(1,5),(2,6),(3,7)
]
for (x1, x2, y1, y2, z1, z2) in bounds_free:
    v_idx = np.array([
        [x1, y1, z1], [x2, y1, z1], [x2, y2, z1], [x1, y2, z1],
        [x1, y1, z2], [x2, y1, z2], [x2, y2, z2], [x1, y2, z2]
    ], dtype=float)
    v = v_idx * scale  # convert to meters

    for a, b in edges:
        all_x_box += [v[a, 0], v[b, 0], None]
        all_y_box += [v[a, 1], v[b, 1], None]
        all_z_box += [v[a, 2], v[b, 2], None]

fig_edges = go.Figure()

fig_edges.add_trace(go.Scatter3d(
    x=all_x_box, y=all_y_box, z=all_z_box,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"FREE cubics ({len(source_boxes_free):,})"
))

if len(centers_free) > 0:
    centers_m = centers_free * scale
    fig_edges.add_trace(go.Scatter3d(
        x=centers_m[:, 0], y=centers_m[:, 1], z=centers_m[:, 2],
        mode="markers",
        marker=dict(size=3, color="royalblue"),
        name="Box centers (FREE)"
    ))

fig_edges.add_trace(go.Scatter3d(
    x=face_x_free, y=face_y_free, z=face_z_free,
    mode="lines",
    line=dict(color="limegreen", width=4),
    name=f"Face-adjacency edges (FREE) ({len(face_pairs_free):,})"
))

fig_edges.update_layout(
    title="FREE boxes: shared-face adjacency (real scale, meters)",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1200,
    height=600
)

fig_edges.show()

print(f"FREE boxes: {len(source_boxes_free)}")
print(f"Face-adjacency pairs: {len(face_pairs_free)}")

In [ ]:
face_pairs_free

## adjacent on the same edge of cubic

In [ ]:
# Xét các cặp cubic tiếp xúc theo 1 cạnh (không phải cùng mặt)

def edge_touch_only(b1, b2, eps=1e-9):
    x1a, x2a, y1a, y2a, z1a, z2a = b1
    x1b, x2b, y1b, y2b, z1b, z2b = b2

    # overlap dương theo từng trục
    ox = overlap_positive(x1a, x2a, x1b, x2b, eps)
    oy = overlap_positive(y1a, y2a, y1b, y2b, eps)
    oz = overlap_positive(z1a, z2a, z1b, z2b, eps)

    # chạm mặt phẳng biên theo từng trục
    tx = (abs(x2a - x1b) <= eps) or (abs(x2b - x1a) <= eps)
    ty = (abs(y2a - y1b) <= eps) or (abs(y2b - y1a) <= eps)
    tz = (abs(z2a - z1b) <= eps) or (abs(z2b - z1a) <= eps)

    # Tiếp xúc theo cạnh:
    # - chạm ở 2 trục, overlap dương ở trục còn lại
    # - loại bỏ trường hợp chạm mặt
    edge_x = ty and tz and ox   # cạnh song song trục X
    edge_y = tx and tz and oy   # cạnh song song trục Y
    edge_z = tx and ty and oz   # cạnh song song trục Z

    return (edge_x or edge_y or edge_z) and (not face_touch(b1, b2, eps))


# Tìm cặp box tiếp xúc cạnh
n = len(source_boxes)
adjacency_edge = {i: set() for i in range(n)}
edge_pairs_edge = []

for i in range(n):
    for j in range(i + 1, n):
        if edge_touch_only(bounds[i], bounds[j]):
            adjacency_edge[i].add(j)
            adjacency_edge[j].add(i)
            edge_pairs_edge.append((i, j))

adjacency_edge = {k: sorted(v) for k, v in adjacency_edge.items()}

print(f"Tổng số box: {n}")
print(f"Số cặp tiếp xúc theo cạnh: {len(edge_pairs_edge)}")
print("5 box đầu tiên và các box tiếp xúc cạnh với nó:")
for i in range(min(5, n)):
    print(f"Box {i} -> {adjacency_edge[i]}")

# Tạo line nối tâm để vẽ nếu cần
edge_x2, edge_y2, edge_z2 = [], [], []
for i, j in edge_pairs_edge:
    c1, c2 = centers[i], centers[j]
    edge_x2 += [c1[0], c2[0], None]
    edge_y2 += [c1[1], c2[1], None]
    edge_z2 += [c1[2], c2[2], None]


In [ ]:
# Plot từ result đã lọc (FREE only): cubic + centers + edge-adjacency (shared edge)
# Theo kích thước thực (meters)

def build_wireframe_arrays(boxes, scale=voxel_size_m, swap_yz=False):
    """
    boxes: list dict có keys 'pos', 'size'
    scale: [dx, dy, dz] (m/voxel)
    swap_yz: nếu True thì đổi trục Y<->Z khi xuất mảng vẽ
    """
    scale = np.asarray(scale, dtype=float)

    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7)
    ]

    xs, ys, zs = [], [], []
    for b in boxes:
        x1, y1, z1 = b["pos"]
        dx, dy, dz = b["size"]
        x2, y2, z2 = x1 + dx, y1 + dy, z1 + dz

        v_idx = np.array([
            [x1, y1, z1], [x2, y1, z1], [x2, y2, z1], [x1, y2, z1],
            [x1, y1, z2], [x2, y1, z2], [x2, y2, z2], [x1, y2, z2]
        ], dtype=float)

        v = v_idx * scale  # index -> meter

        if swap_yz:
            v = v[:, [0, 2, 1]]

        for a, c in edges:
            xs += [v[a, 0], v[c, 0], None]
            ys += [v[a, 1], v[c, 1], None]
            zs += [v[a, 2], v[c, 2], None]

    return xs, ys, zs


source_boxes_free = [to_box_dict(b) for b in results_free] if "results_free" in globals() else [
    to_box_dict(b) for b in results if not b.is_obstacle
]

# Bounds + centers (index-space)
bounds_free = []
centers_free = []
for b in source_boxes_free:
    bd, c = get_bounds_and_center(b)
    bounds_free.append(bd)
    centers_free.append(c)

centers_free = np.array(centers_free, dtype=float) if len(centers_free) else np.empty((0, 3), dtype=float)

# Edge-adjacency trên tập FREE
edge_pairs_edge_free = []
for i in range(len(source_boxes_free)):
    for j in range(i + 1, len(source_boxes_free)):
        if edge_touch_only(bounds_free[i], bounds_free[j]):
            edge_pairs_edge_free.append((i, j))

# Line adjacency (meter-space)
scale = np.asarray(voxel_size_m, dtype=float)
edge_x_free2, edge_y_free2, edge_z_free2 = [], [], []
for i, j in edge_pairs_edge_free:
    c1 = centers_free[i] * scale
    c2 = centers_free[j] * scale
    edge_x_free2 += [c1[0], c2[0], None]
    edge_y_free2 += [c1[1], c2[1], None]
    edge_z_free2 += [c1[2], c2[2], None]

# Wireframe FREE boxes (meter-space)
all_x_box, all_y_box, all_z_box = build_wireframe_arrays(source_boxes_free, scale=voxel_size_m, swap_yz=False)

fig_edges_only = go.Figure()

fig_edges_only.add_trace(go.Scatter3d(
    x=all_x_box, y=all_y_box, z=all_z_box,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"FREE cubics ({len(source_boxes_free):,})"
))

if len(centers_free) > 0:
    centers_m = centers_free * scale
    fig_edges_only.add_trace(go.Scatter3d(
        x=centers_m[:, 0], y=centers_m[:, 1], z=centers_m[:, 2],
        mode="markers",
        marker=dict(size=3, color="royalblue"),
        name="FREE box centers"
    ))

fig_edges_only.add_trace(go.Scatter3d(
    x=edge_x_free2, y=edge_y_free2, z=edge_z_free2,
    mode="lines",
    line=dict(color="magenta", width=4),
    name=f"Edge-adjacency (FREE) ({len(edge_pairs_edge_free):,})"
))

fig_edges_only.update_layout(
    title="FREE cubics tiếp xúc theo 1 cạnh (real scale, meters)",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1000,
    height=600
)

fig_edges_only.show()

print(f"FREE boxes: {len(source_boxes_free)}")
print(f"Edge-adjacency pairs: {len(edge_pairs_edge_free)}")

## adjacent at point

In [ ]:
# Xét các cặp cubic tiếp xúc tại đúng 1 đỉnh (vertex-touch only)

def vertex_touch_only(b1, b2, eps=1e-9):
    x1a, x2a, y1a, y2a, z1a, z2a = b1
    x1b, x2b, y1b, y2b, z1b, z2b = b2

    tx = (abs(x2a - x1b) <= eps) or (abs(x2b - x1a) <= eps)
    ty = (abs(y2a - y1b) <= eps) or (abs(y2b - y1a) <= eps)
    tz = (abs(z2a - z1b) <= eps) or (abs(z2b - z1a) <= eps)

    # Chạm ở cả 3 trục => tiếp xúc tại đỉnh
    return tx and ty and tz and (not face_touch(b1, b2, eps)) and (not edge_touch_only(b1, b2, eps))


def shared_vertex_point(b1, b2, eps=1e-9):
    x1a, x2a, y1a, y2a, z1a, z2a = b1
    x1b, x2b, y1b, y2b, z1b, z2b = b2

    vx = x2a if abs(x2a - x1b) <= eps else (x1a if abs(x1a - x2b) <= eps else None)
    vy = y2a if abs(y2a - y1b) <= eps else (y1a if abs(y1a - y2b) <= eps else None)
    vz = z2a if abs(z2a - z1b) <= eps else (z1a if abs(z1a - z2b) <= eps else None)

    return (vx, vy, vz)


n = len(source_boxes)
adjacency_vertex = {i: set() for i in range(n)}
vertex_pairs = []
vertex_points = []

for i in range(n):
    for j in range(i + 1, n):
        if vertex_touch_only(bounds[i], bounds[j]):
            adjacency_vertex[i].add(j)
            adjacency_vertex[j].add(i)
            vertex_pairs.append((i, j))
            vertex_points.append(shared_vertex_point(bounds[i], bounds[j]))

adjacency_vertex = {k: sorted(v) for k, v in adjacency_vertex.items()}

# unique vertex tiếp xúc
vertex_points_unique = np.unique(np.array(vertex_points, dtype=float), axis=0) if len(vertex_points) > 0 else np.empty((0, 3))

print(f"Tổng số box: {n}")
print(f"Số cặp tiếp xúc tại đỉnh: {len(vertex_pairs)}")
print(f"Số vertex tiếp xúc duy nhất: {len(vertex_points_unique)}")
print("5 box đầu tiên và các box tiếp xúc đỉnh với nó:")
for i in range(min(5, n)):
    print(f"Box {i} -> {adjacency_vertex[i]}")

# Dữ liệu để vẽ line nối tâm (nếu cần)
vertex_x, vertex_y, vertex_z = [], [], []
for i, j in vertex_pairs:
    c1, c2 = centers[i], centers[j]
    vertex_x += [c1[0], c2[0], None]
    vertex_y += [c1[1], c2[1], None]
    vertex_z += [c1[2], c2[2], None]

In [ ]:
# Vẽ các edge nối tâm giữa các cặp box FREE có chung đỉnh (theo kích thước thực: mét)

source_boxes_free = [to_box_dict(b) for b in results_free] if "results_free" in globals() else [
    to_box_dict(b) for b in results if not b.is_obstacle
]

# Bounds + centers cho tập FREE (index-space)
bounds_free = []
centers_free = []
for b in source_boxes_free:
    bd, c = get_bounds_and_center(b)
    bounds_free.append(bd)
    centers_free.append(c)
centers_free = np.array(centers_free, dtype=float)

# Tính các cặp chung đỉnh (vertex-touch only)
vertex_pairs_free = []
for i in range(len(source_boxes_free)):
    for j in range(i + 1, len(source_boxes_free)):
        if vertex_touch_only(bounds_free[i], bounds_free[j]):
            vertex_pairs_free.append((i, j))

# Đổi sang kích thước thực (meter-space)
scale = np.asarray(voxel_size_m, dtype=float)
centers_free_m = centers_free * scale

# Dựng line nối tâm (meter-space)
vertex_x_free, vertex_y_free, vertex_z_free = [], [], []
for i, j in vertex_pairs_free:
    c1, c2 = centers_free_m[i], centers_free_m[j]
    vertex_x_free += [c1[0], c2[0], None]
    vertex_y_free += [c1[1], c2[1], None]
    vertex_z_free += [c1[2], c2[2], None]

# Wireframe box FREE (đã ở meter-space)
all_x_box, all_y_box, all_z_box = build_wireframe_arrays(source_boxes_free, scale=voxel_size_m, swap_yz=False)

fig_vertex = go.Figure()

fig_vertex.add_trace(go.Scatter3d(
    x=all_x_box, y=all_y_box, z=all_z_box,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"FREE cubics ({len(source_boxes_free):,})"
))

if len(centers_free_m) > 0:
    fig_vertex.add_trace(go.Scatter3d(
        x=centers_free_m[:, 0], y=centers_free_m[:, 1], z=centers_free_m[:, 2],
        mode="markers",
        marker=dict(size=3, color="royalblue"),
        name="FREE box centers"
    ))

fig_vertex.add_trace(go.Scatter3d(
    x=vertex_x_free, y=vertex_y_free, z=vertex_z_free,
    mode="lines",
    line=dict(color="gold", width=5),
    name=f"Vertex-adjacency (FREE) ({len(vertex_pairs_free):,})"
))

fig_vertex.update_layout(
    title="Các cặp box FREE có chung đỉnh (real scale, meters)",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1000,
    height=600
)

fig_vertex.show()

## combine

In [ ]:
# Combine 3 loại adjacency (face / edge / vertex) trên tập FREE
# và vẽ kèm wireframe cubic (FREE + OBSTACLE)

# Chuẩn bị box
source_boxes_all = [to_box_dict(b) for b in results]
source_boxes_free = [to_box_dict(b) for b in results_free] if "results_free" in globals() else [
    to_box_dict(b) for b in results if not b.is_obstacle
]
source_boxes_obs = [to_box_dict(b) for b in results if b.is_obstacle]

# Bounds + centers (index-space) cho FREE
bounds_free, centers_free = [], []
for b in source_boxes_free:
    bd, c = get_bounds_and_center(b)
    bounds_free.append(bd)
    centers_free.append(c)
centers_free = np.array(centers_free, dtype=float) if len(centers_free) else np.empty((0, 3), dtype=float)

# Tìm 3 loại adjacency trên FREE
face_pairs_free, edge_pairs_edge_free, vertex_pairs_free = [], [], []
n = len(source_boxes_free)

for i in range(n):
    for j in range(i + 1, n):
        b1, b2 = bounds_free[i], bounds_free[j]
        if face_touch(b1, b2):
            face_pairs_free.append((i, j))
        elif edge_touch_only(b1, b2):
            edge_pairs_edge_free.append((i, j))
        elif vertex_touch_only(b1, b2):
            vertex_pairs_free.append((i, j))

# Helper: cặp index -> mảng line (meter-space)
scale = np.asarray(voxel_size_m, dtype=float)

def pairs_to_line_xyz(pairs, centers_idx, scale_vec):
    x, y, z = [], [], []
    for i, j in pairs:
        c1 = centers_idx[i] * scale_vec
        c2 = centers_idx[j] * scale_vec
        x += [c1[0], c2[0], None]
        y += [c1[1], c2[1], None]
        z += [c1[2], c2[2], None]
    return x, y, z

face_x, face_y, face_z = pairs_to_line_xyz(face_pairs_free, centers_free, scale)
edge_x, edge_y, edge_z = pairs_to_line_xyz(edge_pairs_edge_free, centers_free, scale)
vert_x, vert_y, vert_z = pairs_to_line_xyz(vertex_pairs_free, centers_free, scale)

# Wireframe cubic (meter-space)
x_free_w, y_free_w, z_free_w = build_wireframe_arrays(source_boxes_free, scale=voxel_size_m, swap_yz=False)
x_obs_w, y_obs_w, z_obs_w = build_wireframe_arrays(source_boxes_obs,  scale=voxel_size_m, swap_yz=False)

# Plot
fig_combine = go.Figure()

fig_combine.add_trace(go.Scatter3d(
    x=x_free_w, y=y_free_w, z=z_free_w,
    mode="lines",
    line=dict(color="deepskyblue", width=2),
    name=f"FREE cubics ({len(source_boxes_free):,})"
))

fig_combine.add_trace(go.Scatter3d(
    x=x_obs_w, y=y_obs_w, z=z_obs_w,
    mode="lines",
    line=dict(color="red", width=2),
    name=f"OBSTACLE cubics ({len(source_boxes_obs):,})"
))

if len(centers_free) > 0:
    centers_m = centers_free * scale
    fig_combine.add_trace(go.Scatter3d(
        x=centers_m[:, 0], y=centers_m[:, 1], z=centers_m[:, 2],
        mode="markers",
        marker=dict(size=2.5, color="royalblue"),
        name="FREE centers"
    ))

fig_combine.add_trace(go.Scatter3d(
    x=face_x, y=face_y, z=face_z,
    mode="lines",
    line=dict(color="limegreen", width=4),
    name=f"Face adjacency ({len(face_pairs_free):,})"
))

fig_combine.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines",
    line=dict(color="magenta", width=4),
    name=f"Edge adjacency ({len(edge_pairs_edge_free):,})"
))

fig_combine.add_trace(go.Scatter3d(
    x=vert_x, y=vert_y, z=vert_z,
    mode="lines",
    line=dict(color="gold", width=5),
    name=f"Vertex adjacency ({len(vertex_pairs_free):,})"
))

fig_combine.update_layout(
    title="Combine 3 loại adjacency + wireframe cubic (real scale, meters)",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1200,
    height=700
)

fig_combine.show()

print(f"FREE boxes: {len(source_boxes_free)} | OBSTACLE boxes: {len(source_boxes_obs)}")
print(f"Face pairs: {len(face_pairs_free)}")
print(f"Edge pairs: {len(edge_pairs_edge_free)}")
print(f"Vertex pairs: {len(vertex_pairs_free)}")

In [ ]:
# Lưu toàn bộ edge (face/edge/vertex) vào một mảng dict
# Dùng các biến đã có từ cell trước: centers_free, voxel_size_m,
# face_pairs_free, edge_pairs_edge_free, vertex_pairs_free

scale = np.asarray(voxel_size_m, dtype=float)

edges_data = []

def add_edges(pairs, edge_type):
    for i, j in pairs:
        c1_idx = centers_free[i]
        c2_idx = centers_free[j]
        c1_m = c1_idx * scale
        c2_m = c2_idx * scale

        edges_data.append({
            "type": edge_type,              # "face" | "edge" | "vertex"
            "box_i": int(i),
            "box_j": int(j),
            "center_i_idx": c1_idx.tolist(),
            "center_j_idx": c2_idx.tolist(),
            "center_i_m": c1_m.tolist(),
            "center_j_m": c2_m.tolist(),
            "length_m": float(np.linalg.norm(c2_m - c1_m))
        })

add_edges(face_pairs_free, "face")
add_edges(edge_pairs_edge_free, "edge")
add_edges(vertex_pairs_free, "vertex")

print(f"Total edges saved: {len(edges_data)}")
print("Sample:", edges_data[:3])

In [ ]:
len(edges_data)

## eleminate interscts obstacle

In [ ]:
# Lọc các edge bị cắt qua OBSTACLE cubic để tạo edges_data_nocut

def segment_intersects_aabb(p0, p1, bmin, bmax, eps=1e-9):
	"""
	Kiểm tra đoạn thẳng p0->p1 có giao với AABB [bmin, bmax] hay không (slab method).
	p0, p1, bmin, bmax: np.array shape (3,)
	"""
	d = p1 - p0
	tmin, tmax = 0.0, 1.0

	for k in range(3):
		if abs(d[k]) < eps:
			# Đoạn song song trục k -> phải nằm trong slab theo trục k
			if p0[k] < bmin[k] - eps or p0[k] > bmax[k] + eps:
				return False
		else:
			inv_d = 1.0 / d[k]
			t1 = (bmin[k] - p0[k]) * inv_d
			t2 = (bmax[k] - p0[k]) * inv_d
			if t1 > t2:
				t1, t2 = t2, t1
			tmin = max(tmin, t1)
			tmax = min(tmax, t2)
			if tmin > tmax:
				return False

	return True


# Lấy edges source (phòng trường hợp bạn đặt tên edge_data thay vì edges_data)
source_edges = edges_data if "edges_data" in globals() else edge_data

# Obstacle boxes -> AABB theo meter-space (cùng hệ với center_i_m/center_j_m)
scale = np.asarray(voxel_size_m, dtype=float)
obs_boxes = [b for b in results if b.is_obstacle]

obs_aabbs_m = []
for b in obs_boxes:
	bmin = np.array([b.x_rng[0], b.y_rng[0], b.z_rng[0]], dtype=float) * scale
	bmax = np.array([b.x_rng[1], b.y_rng[1], b.z_rng[1]], dtype=float) * scale
	obs_aabbs_m.append((bmin, bmax))

# Lọc edge không cắt obstacle
edges_data_nocut = []
cut_count = 0

for e in source_edges:
	p0 = np.asarray(e["center_i_m"], dtype=float)
	p1 = np.asarray(e["center_j_m"], dtype=float)

	is_cut = False
	for bmin, bmax in obs_aabbs_m:
		if segment_intersects_aabb(p0, p1, bmin, bmax):
			is_cut = True
			break

	if is_cut:
		cut_count += 1
	else:
		edges_data_nocut.append(e)

print(f"Total edges input: {len(source_edges)}")
print(f"Edges bị cắt bởi obstacle: {cut_count}")
print(f"Edges sau khi lọc (edges_data_nocut): {len(edges_data_nocut)}")

In [ ]:
edges_data_nocut

In [ ]:
# Vẽ lại 3D với các edge đã lọc: edges_data_nocut

if "edges_data_nocut" not in globals():
    raise ValueError("Chưa có edges_data_nocut. Hãy chạy cell lọc edge trước.")

fig_filtered = go.Figure()

# (Tuỳ chọn) thêm wireframe FREE/OBSTACLE nếu đã có sẵn
if all(k in globals() for k in ["x_free_w", "y_free_w", "z_free_w"]):
    fig_filtered.add_trace(go.Scatter3d(
        x=x_free_w, y=y_free_w, z=z_free_w,
        mode="lines",
        line=dict(color="deepskyblue", width=2),
        name="FREE cubics"
    ))

if all(k in globals() for k in ["x_obs_w", "y_obs_w", "z_obs_w"]):
    fig_filtered.add_trace(go.Scatter3d(
        x=x_obs_w, y=y_obs_w, z=z_obs_w,
        mode="lines",
        line=dict(color="red", width=2),
        name="OBSTACLE cubics"
    ))

# Gom edge theo type để tô màu
type_style = {
    "face": dict(color="limegreen", width=4),
    "edge": dict(color="magenta", width=4),
    "vertex": dict(color="gold", width=5)
}

for t in ["face", "edge", "vertex"]:
    ex, ey, ez = [], [], []
    count = 0

    for e in edges_data_nocut:
        if e.get("type") != t:
            continue
        p0 = e["center_i_m"]
        p1 = e["center_j_m"]
        ex += [p0[0], p1[0], None]
        ey += [p0[1], p1[1], None]
        ez += [p0[2], p1[2], None]
        count += 1

    if count > 0:
        fig_filtered.add_trace(go.Scatter3d(
            x=ex, y=ey, z=ez,
            mode="lines",
            line=type_style[t],
            name=f"{t} edges (filtered) ({count:,})"
        ))

fig_filtered.update_layout(
    title=f"Filtered edges (no cut through obstacles) - total: {len(edges_data_nocut):,}",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1200,
    height=700
)

fig_filtered.show()

In [ ]:
# Vẽ ONLY các edge bị cut + wireframe voxel/cubic trong không gian 3D

# 1) Hàm kiểm tra giao đoạn thẳng với AABB (nếu chưa có)
if "segment_intersects_aabb" not in globals():
    def segment_intersects_aabb(p0, p1, bmin, bmax, eps=1e-9):
        d = p1 - p0
        tmin, tmax = 0.0, 1.0
        for k in range(3):
            if abs(d[k]) < eps:
                if p0[k] < bmin[k] - eps or p0[k] > bmax[k] + eps:
                    return False
            else:
                inv_d = 1.0 / d[k]
                t1 = (bmin[k] - p0[k]) * inv_d
                t2 = (bmax[k] - p0[k]) * inv_d
                if t1 > t2:
                    t1, t2 = t2, t1
                tmin = max(tmin, t1)
                tmax = min(tmax, t2)
                if tmin > tmax:
                    return False
        return True

# 2) Chuẩn bị dữ liệu edge nguồn
source_edges = edges_data if "edges_data" in globals() else []
if len(source_edges) == 0:
    raise ValueError("Không tìm thấy edges_data. Hãy chạy cell tạo edges_data trước.")

# 3) Obstacle AABB trong meter-space
scale = np.asarray(voxel_size_m, dtype=float)
obs_boxes = [b for b in results if b.is_obstacle]
obs_aabbs_m = []
for b in obs_boxes:
    bmin = np.array([b.x_rng[0], b.y_rng[0], b.z_rng[0]], dtype=float) * scale
    bmax = np.array([b.x_rng[1], b.y_rng[1], b.z_rng[1]], dtype=float) * scale
    obs_aabbs_m.append((bmin, bmax))

# 4) Lấy ONLY các edge bị cut
edges_data_cut = []
for e in source_edges:
    p0 = np.asarray(e["center_i_m"], dtype=float)
    p1 = np.asarray(e["center_j_m"], dtype=float)

    is_cut = False
    for bmin, bmax in obs_aabbs_m:
        if segment_intersects_aabb(p0, p1, bmin, bmax):
            is_cut = True
            break

    if is_cut:
        edges_data_cut.append(e)

print(f"Total edges: {len(source_edges)}")
print(f"Cut edges: {len(edges_data_cut)}")

# 5) Plot
fig_cut = go.Figure()

# Wireframe FREE/OBSTACLE (nếu đã có sẵn từ cell trước)
if all(k in globals() for k in ["x_free_w", "y_free_w", "z_free_w"]):
    fig_cut.add_trace(go.Scatter3d(
        x=x_free_w, y=y_free_w, z=z_free_w,
        mode="lines",
        line=dict(color="deepskyblue", width=2),
        name="FREE cubics"
    ))

if all(k in globals() for k in ["x_obs_w", "y_obs_w", "z_obs_w"]):
    fig_cut.add_trace(go.Scatter3d(
        x=x_obs_w, y=y_obs_w, z=z_obs_w,
        mode="lines",
        line=dict(color="red", width=2),
        name="OBSTACLE cubics"
    ))

# Edge bị cut theo từng loại
type_style = {
    "face": dict(color="limegreen", width=4),
    "edge": dict(color="magenta", width=4),
    "vertex": dict(color="gold", width=5)
}

for t in ["face", "edge", "vertex"]:
    ex, ey, ez = [], [], []
    cnt = 0
    for e in edges_data_cut:
        if e.get("type") != t:
            continue
        p0 = e["center_i_m"]
        p1 = e["center_j_m"]
        ex += [p0[0], p1[0], None]
        ey += [p0[1], p1[1], None]
        ez += [p0[2], p1[2], None]
        cnt += 1

    if cnt > 0:
        fig_cut.add_trace(go.Scatter3d(
            x=ex, y=ey, z=ez,
            mode="lines",
            line=type_style[t],
            name=f"{t} edges CUT ({cnt:,})"
        ))

fig_cut.update_layout(
    title=f"Only CUT edges in 3D (with voxel/cubic wireframes) - total cut: {len(edges_data_cut):,}",
    scene=dict(
        aspectmode="data",
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=1200,
    height=700
)

fig_cut.show()

## remark
source_boxes_free: list free-fly cubic in 3d Space (no obstacle) <br>
edges_data_nocut: list valid undirected edge in 3d Space


In [ ]:
source_boxes_free

In [ ]:
edges_data_nocut

# Energy calculation

In [ ]:
def calculate_drone_power(v_i, w_i, a_i, params):
    """
    Calculate total power consumption Pi according to the formula sequence.
    v_i, w_i, a_i are numpy array vectors with shape (3,)
    """
    # Get constants from params
    rho = params['rho']       # Air density
    Cd = params['Cd']         # Drag coefficient
    Af = params['Af']         # Frontal area
    m = params['m']           # Drone weight
    g_acc = params['g_acc']   # Gravitational acceleration vector [0, 0, -9.81]
    A = params['A']           # Rotor disk area
    Pp_hover = params['Pp_hover'] # Profile power when hovering
    mg_scalar = m * np.linalg.norm(g_acc)

    # (2) Velocity relative to wind
    v_a = v_i - w_i
    v_a_norm = np.linalg.norm(v_a)

    # (3) Aerodynamic drag
    Di = -0.5 * rho * Cd * Af * v_a_norm * v_a

    # (4) Thrust force vector
    # print("a_i:", np.array(a_i))
    Ti_vec = m * np.array(a_i) - m * g_acc - Di

    # (5) Magnitude of thrust force
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Unit direction vector of thrust
    ti_hat = Ti_vec / Ti_mag if Ti_mag != 0 else np.array([0, 0, 1])

    # (7) Velocity component relative to thrust axis (scalar)
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity (scalar)
    # Formula: v_ind = -1/2*vc + sqrt((1/2*vc)^2 + Ti/(2*rho*A))
    term1 = -0.5 * vc_i
    term2 = np.sqrt((0.5 * vc_i)**2 + Ti_mag / (2 * rho * A))
    v_ind = term1 + term2

    # (9) Useful power
    Pu_i = np.dot(Ti_vec, v_a)

    # (10) Induced power
    Pind_i = Ti_mag * v_ind

    # (11) Profile power
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5

    # (12) Total power
    Pi = Pu_i + Pind_i + Pp_i

    return {
        "Pi": Pi,
        "Pu_i": Pu_i,
        "Pind_i": Pind_i,
        "Pp_i": Pp_i,
        "Ti_vec": Ti_vec,
        "Ti_mag": Ti_mag,
    }

In [ ]:
def generate_directed_edge_dict(box_data, edge_data,  parameters):
    """
    Create a 3D directed edge dictionary with:
    - box_data: {
    "pos": (x1, y1, z1),
    "size": (dx, dy, dz),
    "avg_wind": np.array([wx, wy, wz]) hoặc list 3 phần tử,
    "turbulence_level": float
    }
    - edge_data: {
    "type": "face | edge | vertex",
    "box_i": 12,
    "box_j": 57,
    "center_i_idx": [xi, yi, zi],
    "center_j_idx": [xj, yj, zj],
    "center_i_m": [Xi, Yi, Zi],
    "center_j_m": [Xj, Yj, Zj],
    "length_m": l
    }
    - parameters: dict with keys 'wind_field', 'cell_size'  (in meters), 'drone_params' (dict for power calculation)

    """
    moore_dict = {}
    for edge in edge_data:
        V_i = box_data[edge["box_i"]]["size"][0] * box_data[edge["box_i"]]["size"][1] * box_data[edge["box_i"]]["size"][2]
        V_j = box_data[edge["box_j"]]["size"][0] * box_data[edge["box_j"]]["size"][1] * box_data[edge["box_j"]]["size"][2]

        w_i = (V_i * box_data[edge["box_i"]]["avg_wind"] + V_j * box_data[edge["box_j"]]["avg_wind"]) / (V_i + V_j)
        for speed_level in speed_map.keys():
            speed=speed_map[speed_level]
            v_i = (speed / edge["length_m"]) * (np.array(edge["center_j_m"]) - np.array(edge["center_i_m"]))

            power_result = calculate_drone_power(v_i, w_i, np.array([0.0, 0.0, 0.0]), parameters)

            edge_key = f"{edge['box_i']}_{edge['box_j']}_{speed_level}"
            moore_dict[edge_key] = {
                "startbox_idx": edge["box_i"],
                "endbox_idx": edge["box_j"],
                "startpoint": edge["center_i_idx"],
                "startpoint_m": edge["center_i_m"],
                "endpoint": edge["center_j_idx"],
                "endpoint_m": edge["center_j_m"],
                "euclidean_distance": edge["length_m"],
                "v_level": speed_level ,
                "v": np.linalg.norm(v_i),
                "v_vector": v_i,
                "wind_vector": w_i,
                "Pi": power_result["Pi"],
                "Tmaneuver": power_result["Ti_mag"],
                "Energy": power_result["Pi"] * (edge["length_m"] / speed),
                "length_m": edge["length_m"],
            }

            v_i_reverse = -v_i
            power_result_reverse = calculate_drone_power(v_i_reverse, w_i, np.array([0.0, 0.0, 0.0]), parameters)

            edge_key_reverse = f"{edge['box_j']}_{edge['box_i']}_{speed_level}"
            moore_dict[edge_key_reverse] = {
                "startbox_idx": edge["box_j"],
                "endbox_idx": edge["box_i"],
                "startpoint": edge["center_j_idx"],
                "startpoint_m": edge["center_j_m"],
                "endpoint": edge["center_i_idx"],
                "endpoint_m": edge["center_i_m"],
                "euclidean_distance": edge["length_m"],
                "v_level": speed_level,
                "v": np.linalg.norm(v_i_reverse),
                "v_vector": v_i_reverse,
                "wind_vector": w_i,
                "Pi": power_result_reverse["Pi"],
                "Tmaneuver": power_result_reverse["Ti_mag"],
                "Energy": power_result_reverse["Pi"] * (edge["length_m"] / speed),
                "length_m": edge["length_m"],
            }

    return moore_dict

In [ ]:
def calculate_energy_transition(edge_in_key, edge_out_key, wind, params,dt):
    """
    Calculate energy at the intersection point between 2 edges based on Vlevel
    """
    # 1. Decode edge information
    # edge_in_key=space_graph[edge_in_key]
    # edge_out_key=space_graph[edge_out_key]
    p_prev, p_curr, v_in_vec,v_mag_in = edge_in_key["startpoint"], edge_in_key["endpoint"], edge_in_key["v_vector"],edge_in_key["v"]
    _, p_next, v_out_vec,v_mag_out = edge_out_key["startpoint"], edge_out_key["endpoint"], edge_out_key["v_vector"],edge_out_key["v"]

  
    # 3. Calculate v_i and a_i at point p_curr
    # Assume dt is the transition time (average distance / average velocity)
    # dist_avg = (np.linalg.norm(p_curr - p_prev) + np.linalg.norm(p_next - p_curr)) / 2
    # dt = dist_avg / ((v_mag_in + v_mag_out) / 2)

    v_i = (v_in_vec + v_out_vec) / 2
    a_i = (v_out_vec - v_in_vec) / dt

    # --- Physical sequence ---
    rho, Cd, Af, m, A = params['rho'], params['Cd'], params['Af'], params['m'], params['A']
    g_vec = params['g_acc']
    Pp_hover = params['Pp_hover']
    mg_scalar = m * np.linalg.norm(g_vec)

    # (2) Relative velocity
    v_a = v_i - wind
    va_norm = np.linalg.norm(v_a)

    # (3) Drag force Di
    Di = -0.5 * rho * Cd * Af * va_norm * v_a

    # (4) Thrust force Ti (Vector)
    Ti_vec = m * a_i - m * g_vec - Di
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Thrust direction
    ti_hat = Ti_vec / Ti_mag if Ti_mag > 1e-6 else np.array([0, 0, 1])

    # (7) Velocity along thrust axis
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity
    v_ind = -0.5 * vc_i + np.sqrt(max(0, (0.5 * vc_i)**2 + Ti_mag / (2 * rho * A)))

    # (9-12) Power components
    Pu_i = np.dot(Ti_vec, v_a)
    Pind_i = Ti_mag * v_ind
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5
    Pi = Pu_i + Pind_i + Pp_i

    return {
        # "edge_in": edge_in_key,
        # "edge_out": edge_out_key,
        "total_power_Pi": Pi,
        "Tmaneuver": Ti_mag,
        # "acceleration": a_i
    }

In [ ]:
def find_box_index(point, box_data):
    point = np.array(point, dtype=float)

    mins = np.array([box["pos_m"] for box in box_data], dtype=float)
    sizes = np.array([box["size_m"] for box in box_data], dtype=float)
    maxs = mins + sizes

    # tolerance tránh lỗi float
    eps = 1e-9

    inside = np.all((point >= mins - eps) & (point < maxs + eps), axis=1)

    indices = np.where(inside)[0]

    if len(indices) == 0:
        return None
    elif len(indices) == 1:
        return indices[0]
    else:
        # nếu overlap → chọn box nhỏ nhất
        volumes = sizes[indices].prod(axis=1)
        return indices[np.argmin(volumes)]
    


In [ ]:
space_graph = generate_directed_edge_dict(source_boxes_free, edges_data_nocut, drone_params)

In [ ]:
space_graph

In [ ]:
len(space_graph)

In [ ]:
start_point = [250, 40, 0]      # meters
start_box_idx = find_box_index(start_point, source_boxes_free)
print(f"Start point {start_point} in box index: {start_box_idx}")
end_point=[1000,1100,0]
end_box_idx = find_box_index(end_point, source_boxes_free)
print(f"End point {end_point} in box index: {end_box_idx}")

In [ ]:
source_boxes_free

In [ ]:
import numpy as np
import plotly.graph_objects as go

# =========================================
# 1. DRAW WIREFRAME CUBIC
# =========================================
def build_wireframe_arrays(boxes, scale=None, swap_yz=False):
    """
    Vẽ wireframe các boxes theo kích thước thật
    boxes: list dict có keys 'pos', 'size'
    scale: [dx, dy, dz] (m/voxel)
    swap_yz: nếu True thì đổi trục Y<->Z
    """
    if scale is None:
        scale = np.array([1.0, 1.0, 1.0])
    scale = np.asarray(scale, dtype=float)

    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7)
    ]

    xs, ys, zs = [], [], []
    for b in boxes:
        x1, y1, z1 = b["pos"]
        dx, dy, dz = b["size"]
        x2, y2, z2 = x1 + dx, y1 + dy, z1 + dz

        v_idx = np.array([
            [x1, y1, z1], [x2, y1, z1], [x2, y2, z1], [x1, y2, z1],
            [x1, y1, z2], [x2, y1, z2], [x2, y2, z2], [x1, y2, z2]
        ], dtype=float)

        v = v_idx * scale  # index -> meter

        if swap_yz:
            v = v[:, [0, 2, 1]]

        for a, c in edges:
            xs += [v[a, 0], v[c, 0], None]
            ys += [v[a, 1], v[c, 1], None]
            zs += [v[a, 2], v[c, 2], None]

    return xs, ys, zs


# =========================================
# 2. MAIN PLOT FUNCTION
# =========================================
fig = go.Figure()

# ---------- 1. DRAW WIREFRAME ALL CUBICS ----------
xs_wf, ys_wf, zs_wf = build_wireframe_arrays(source_boxes_free, scale=voxel_size_m, swap_yz=False)

fig.add_trace(go.Scatter3d(
    x=xs_wf, y=ys_wf, z=zs_wf,
    mode="lines",
    line=dict(color="deepskyblue", width=1.5),
    name=f"Cubic wireframes ({len(source_boxes_free)})"
))

# ---------- 2. DRAW FILTERED EDGES ----------
ex_edge, ey_edge, ez_edge = [], [], []
for edge in edges_data_nocut:
    p_i = edge["center_i_m"]
    p_j = edge["center_j_m"]
    ex_edge += [p_i[0], p_j[0], None]
    ey_edge += [p_i[1], p_j[1], None]
    ez_edge += [p_i[2], p_j[2], None]

fig.add_trace(go.Scatter3d(
    x=ex_edge, y=ey_edge, z=ez_edge,
    mode="lines",
    line=dict(color="orange", width=2),
    name=f"Filtered Edges ({len(edges_data_nocut)})"
))

# ---------- 3. DRAW START BOX ----------
if start_box_idx is not None:
    start_box = source_boxes_free[start_box_idx]
    xs_start, ys_start, zs_start = build_wireframe_arrays([start_box], scale=voxel_size_m, swap_yz=False)
    
    fig.add_trace(go.Scatter3d(
        x=xs_start, y=ys_start, z=zs_start,
        mode="lines",
        line=dict(color="limegreen", width=4),
        name=f"Start Box (idx={start_box_idx})"
    ))

# ---------- 4. DRAW END BOX ----------
if end_box_idx is not None:
    end_box = source_boxes_free[end_box_idx]
    xs_end, ys_end, zs_end = build_wireframe_arrays([end_box], scale=voxel_size_m, swap_yz=False)
    
    fig.add_trace(go.Scatter3d(
        x=xs_end, y=ys_end, z=zs_end,
        mode="lines",
        line=dict(color="red", width=4),
        name=f"End Box (idx={end_box_idx})"
    ))

# ---------- 5. MARK START/END POINTS ----------
fig.add_trace(go.Scatter3d(
    x=[start_point[0], end_point[0]],
    y=[start_point[1], end_point[1]],
    z=[start_point[2], end_point[2]],
    mode="markers+text",
    marker=dict(size=10, color=["green", "red"]),
    text=["START", "END"],
    textposition="top center",
    textfont=dict(size=12, color="black"),
    name="Start/End Points"
))

# ---------- 6. SET LAYOUT ----------
fig.update_layout(
    title=f"Full 3D Environment: {len(source_boxes_free)} Cubics + {len(edges_data_nocut)} Edges",
    scene=dict(
        xaxis=dict(title="X (m)"),
        yaxis=dict(title="Y (m)"),
        zaxis=dict(title="Z (m)"),
        aspectmode="data"
    ),
    width=1200,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()

In [ ]:
# Filter all edges starting from point (0, 0, 0)
edges_from_origin = {key: value for key, value in space_graph.items() if value['startbox_idx'] == start_box_idx}

print(f"Number of edges starting: {len(edges_from_origin)}")
for key, data in edges_from_origin.items():

    astart = data['v_vector']/dt_takeoff
    Etakeoff = calculate_drone_power(data['v_vector']/2, source_boxes_free[data["startbox_idx"]]["avg_wind"], astart, drone_params)["Pi"]*dt_takeoff
    space_graph[key]["Etakeoff"] = Etakeoff  # Add takeoff energy to the edge
    print(f"{key}:   Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Etakeoff={Etakeoff:.2f} J")

In [ ]:
# Filter all edges ending at point (4, 4, 5)
edges_to_destination = {key: value for key, value in space_graph.items() if value['endbox_idx'] == end_box_idx}

print(f"Number of edges ending: {len(edges_to_destination)}")
print("\nEdges to destination:")
for key, data in edges_to_destination.items():
    astart = data['v_vector']/dt_landing
    Elanding = calculate_drone_power(data['v_vector']/2, source_boxes_free[data["endbox_idx"]]["avg_wind"], astart, drone_params)["Pi"]*dt_landing
    space_graph[key]["Elanding"] = Elanding  # Add landing energy to the edge
    print(f"{key}: {data['endpoint']}  Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Elanding={Elanding:.2f} J")

##  Eliminate variable

In [ ]:
# Eliminate edges that end at start_point or start at end_point
edges_to_start = {
    key: data
    for key, data in space_graph.items()
    if data["endbox_idx"] == start_box_idx
}

edges_from_end = {
    key: data
    for key, data in space_graph.items()
    if data["startbox_idx"] == end_box_idx
}

print(len(edges_to_start))
print(len(edges_from_end))

In [ ]:
edges_exceeded_Tmax={
    key: data
    for key, data in space_graph.items()
    if data["Tmaneuver"] > Tmax
}
print(len(edges_exceeded_Tmax))

In [ ]:
# Remove edges in space_graph whose key is in edges_to_start or edges_from_end
keys_to_remove = set(edges_to_start.keys()) | set(edges_from_end.keys() | set(edges_exceeded_Tmax.keys()))

removed = 0
for k in keys_to_remove:
    if k in space_graph:
        del space_graph[k]
        removed += 1

print(f"Deleted {removed} keys from space_graph")
print(f"Remaining edges: {len(space_graph)}")

# QUBO matrix creation

In [ ]:
# Create QUBO matrix from space_graph
edge_keys = list(space_graph.keys())
N = len(edge_keys)
edge_index = {key: idx for idx, key in enumerate(edge_keys)}

print(f"Number of edges (N): {N}")
print(f"QUBO matrix size: {N} x {N}")

# Initialize QUBO matrix
Q = np.zeros((N, N), dtype=float)


## Objective function

In [ ]:
# Assign diagonal values Q[i,i] = 3 for edges with the startpoint
list_edges_start = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["startbox_idx"] == start_box_idx
}
max_energy_start = 0
for key in list_edges_start:
    idx = edge_index[key]  # get key index in graph
    bufVal= 0.5*space_graph[key]["Energy"] + space_graph[key].get("Etakeoff", 0.0)
    Q[idx, idx] += bufVal/1000  # Scale down energy to fit into QUBO range
    print(f"Edge {key}: Energy={space_graph[key]['Energy']:.2f} J, Etakeoff={space_graph[key].get('Etakeoff', 0.0):.2f} J, Q[{idx},{idx}]={Q[idx, idx]:.2f}")
    if bufVal > max_energy_start:
        max_energy_start = bufVal

print(f"Max energy for edges from start point: {max_energy_start:.2f} J")


In [ ]:
# Assign diagonal values Q[i,i] = 3 for edges with the endpoint
list_edges_end = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["endbox_idx"] == end_box_idx
}
max_energy_end = 0

for key in list_edges_end:
    idx = edge_index[key]  # get key index in graph
    bufVal = 0.5*space_graph[key]["Energy"] + space_graph[key].get("Elanding", 0.0)
    Q[idx, idx] += bufVal/1000  # Scale down energy to fit into QUBO range 
    print(f"Edge {key}: Energy={space_graph[key]['Energy']:.2f} J, Elanding={space_graph[key].get('Elanding', 0.0):.2f} J, Q[{idx},{idx}]={Q[idx, idx]:.2f}")
    if bufVal > max_energy_end:
        max_energy_end = bufVal
print(f"Max energy for edges to end point: {max_energy_end:.2f} J")

In [ ]:
Q

In [ ]:
import itertools
# Loop through all points in space, excluding start and end

all_mid_boxes = []
for  box_idx, box in enumerate(source_boxes_free):
    if box_idx == start_box_idx or box_idx == end_box_idx: 
        continue
    all_mid_boxes.append(box_idx)
print(f"Total mid boxes: {len(all_mid_boxes)}")


In [ ]:
max_pair_energy = 0
max_pair_thrust = 0
for box_idx in all_mid_boxes:
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if v["startbox_idx"] == box_idx
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if v["endbox_idx"] == box_idx
    }
    print(f"Box {box_idx}: {len(edges_in_list)} incoming edges, {len(edges_out_list)} outgoing edges")
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]

            V_i = source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][0] * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][1] * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][2]
            V_j = source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][0] * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][1] * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][2]

            w_i = (V_i * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["avg_wind"] + V_j * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["avg_wind"]) / (V_i + V_j)

            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], w_i, drone_params, dt)
            bufVal = (0.5*space_graph[edges_in]["Energy"] + 0.5*space_graph[edges_out]["Energy"] + change_dir["total_power_Pi"]*(dt))/1000  # Scale down to kJ for QUBO
            Q[min(idx_in, idx_out), max(idx_in, idx_out)] += bufVal  # Convert J to kJ for better scaling
            if bufVal > max_pair_energy:
                max_pair_energy = bufVal
            if change_dir["Tmaneuver"] > max_pair_thrust:
                max_pair_thrust = change_dir["Tmaneuver"]
            print(f"Pair: {edges_in}(id: {idx_in}) -> {edges_out}(id: {idx_out}), Transition Energy={change_dir['total_power_Pi']:.2f} J, Total Pair Energy={bufVal:.2f} J, Thrust={change_dir['Tmaneuver']:.2f} N")

print(f"Max pair energy (diagonal + transition): {max_pair_energy:.2f} J")
print(f"Max pair thrust power: {max_pair_thrust:.2f} N")

In [ ]:
Q

## Constraint

In [ ]:
for key in list_edges_start:
    idx = edge_index[key]  # get key index in graph
    Q[idx, idx] -= max_energy_start

for i in range(len(list_edges_start)):
    for j in range(i+1, len(list_edges_start)):
        idx_i = edge_index[list(list_edges_start.keys())[i]]
        idx_j = edge_index[list(list_edges_start.keys())[j]]

        Q[min(idx_i, idx_j), max(idx_i, idx_j)] += max_energy_start

In [ ]:
for key in list_edges_end:
    idx = edge_index[key]  # get key index in graph
    Q[idx, idx] -= max_energy_end

for i in range(len(list_edges_end)):
    for j in range(i+1, len(list_edges_end)):
        idx_i = edge_index[list(list_edges_end.keys())[i]]
        idx_j = edge_index[list(list_edges_end.keys())[j]]

        Q[min(idx_i, idx_j), max(idx_i, idx_j)] += max_energy_end

In [ ]:
for box_idx in all_mid_boxes:

    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if v["startbox_idx"] == box_idx
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if v["endbox_idx"] == box_idx
    }
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        Q[idx_in, idx_in] += 10*max_pair_energy

    for edges_out in edges_out_list:
        idx_out = edge_index[edges_out]
        Q[idx_out, idx_out] += 10*max_pair_energy

    for i in range(len(edges_in_list)):
        for j in range(i+1, len(edges_in_list)):
            idx_i = edge_index[list(edges_in_list.keys())[i]]
            idx_j = edge_index[list(edges_in_list.keys())[j]]

            Q[min(idx_i, idx_j), max(idx_i, idx_j)] += 20*max_pair_energy

    for i in range(len(edges_out_list)):
        for j in range(i+1, len(edges_out_list)):
            idx_i = edge_index[list(edges_out_list.keys())[i]]
            idx_j = edge_index[list(edges_out_list.keys())[j]]

            Q[min(idx_i, idx_j), max(idx_i, idx_j)] += 20*max_pair_energy

    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            Q[min(idx_in, idx_out), max(idx_in, idx_out)] +=-20*max_pair_energy

In [ ]:
num_violations = 0
for box_idx in all_mid_boxes:
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if v["startbox_idx"] == box_idx
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if v["endbox_idx"] == box_idx
    }
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            V_i = source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][0] * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][1] * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["size"][2]
            V_j = source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][0] * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][1] * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["size"][2]

            w_i = (V_i * source_boxes_free[space_graph[edges_in]["startbox_idx"]]["avg_wind"] + V_j * source_boxes_free[space_graph[edges_out]["endbox_idx"]]["avg_wind"]) / (V_i + V_j)

            idx_out = edge_index[edges_out]
            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], w_i, drone_params, dt)
            if change_dir["Tmaneuver"] > Tmax:
                print(f"Violation: Transition from {edges_in} to {edges_out} has Tmaneuver={change_dir['Tmaneuver']:.2f} N > Tmax={Tmax} N")
                Q[min(idx_in, idx_out), max(idx_in, idx_out)] += 10*max_pair_energy
                num_violations += 1
print(f"Number of pairs with Tmaneuver > Tmax: {num_violations}")

In [ ]:
Q

In [ ]:
import numpy as np
import pandas as pd



Q_np = np.asarray(Q)

df_Q = pd.DataFrame(Q_np)
csv_path = "Q_matrix.csv"
df_Q.to_csv(csv_path, index=False)

print(f"Da tao DataFrame tu Q voi shape: {df_Q.shape}")
print(f"Da luu CSV: {csv_path}")
df_Q.head()

# Run

In [ ]:
print(f"\nQUBO Matrix Q shape: {Q.shape}")
print(f"Non-zero elements: {np.count_nonzero(Q)}")
print(f"Diagonal sample (first 5): {np.diag(Q)[:5]}")

In [ ]:
offset = max(max_energy_start, max_energy_end, max_pair_energy) + 10

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import time
# --- 1. SETUP LICENSE ---
try:
    # Khởi tạo môi trường trống và nạp file license từ đường dẫn cụ thể
    env = gp.Env(empty=True)
    env.readParams(r"C:\Users\dungdv\gurobi.lic")
    env.start()
    print(">>> Đã nhận License thành công.")
except gp.GurobiError:
    print(">>> Lỗi: Không đọc được file License. Kiểm tra lại đường dẫn ổ D.")
    exit()

# Giả sử đây là ma trận Q của Dũng (thay bằng ma trận thực tế của bạn)
n = Q.shape[0]
print(f"QUBO matrix Q size: {Q.shape}")
# --- 3. XÂY DỰNG MODEL ---
model = gp.Model("QUBO_Dung", env=env)
# Thêm biến nhị phân x (x_i in {0, 1})
x = model.addMVar(n, vtype=GRB.BINARY, name="x")

# Thiết lập hàm mục tiêu: Minimize x^T * Q * x
model.setObjective(x @ Q @ x, GRB.MINIMIZE)

# QUBO thường là Non-Convex (không lồi), cần tham số này để Gurobi giải được
model.setParam('NonConvex', 2)
model.setParam("Threads", 14)

def my_early_stop(model, where):
    # Chúng ta chỉ quan tâm khi Gurobi đang giải bài toán nguyên (MIP)
    if where == GRB.Callback.MIP:
        
        # 1. Lấy thông tin thực tế từ Solver
        runtime = model.cbGet(GRB.Callback.RUNTIME)      # Thời gian đã chạy
        obj_best = model.cbGet(GRB.Callback.MIP_OBJBST)  # Nghiệm tốt nhất (Incumbent)
        bound_best = model.cbGet(GRB.Callback.MIP_OBJBND) # Cận tốt nhất (Best Bound)
        
        # Tính Gap hiện tại (%)
        if abs(obj_best) < 1e-6: # Tránh chia cho 0
            current_gap = float('inf')
        else:
            current_gap = abs(bound_best - obj_best) / abs(obj_best) * 100

        # 2. Khởi tạo các biến "bộ nhớ" để theo dõi sự tiến triển
        if not hasattr(model, "_last_incumbent"):
            model._last_incumbent = obj_best
            model._last_improvement_time = runtime
            model._patience = 300*3  # Đợi tối đa 300 giây (5 phút) không có nghiệm mới thì dừng

        # 3. KIỂM TRA: Nếu tìm thấy nghiệm mới tốt hơn (Early Stop cho Incumbent)
        if abs(obj_best - model._last_incumbent) > 1e-6:
            model._last_incumbent = obj_best
            model._last_improvement_time = runtime
            print(f"--> Tìm thấy nghiệm mới: {obj_best:.2f} tại giây thứ {runtime:.1f}")

        # 4. QUYẾT ĐỊNH: Nếu quá thời gian 'patience' mà không có gì mới
        if runtime - model._last_improvement_time > model._patience:
            print(f"!!! Early Stopping: Đã {model._patience}s không có nghiệm mới tốt hơn. Dừng tại đây!")
            model.terminate()
        
# --- 4. GIẢI ---
print(">>> Đang bắt đầu giải QUBO...")
start_time = time.time()

model.optimize(my_early_stop)

# --- 5. XUẤT KẾT QUẢ ---
if model.status == GRB.OPTIMAL:
    print("\n" + "="*30)
    print(f"KẾT QUẢ TỐI ƯU CHO DŨNG:")
    print(f"Vector x: {x.X}")
    print(f"Giá trị năng lượng cực tiểu (Objective): {model.ObjVal}")
    print("="*30)
else:
    print(f"Không tìm thấy lời giải tối ưu. Trạng thái: {model.status}")
print(f"Thời gian giải: {time.time() - start_time:.2f} giây")
# Đóng môi trường sau khi xong
env.dispose()

In [ ]:
x.X

In [ ]:
x.X.sum()

In [ ]:
import numpy as np
import plotly.graph_objects as go


def plot_space_graph_with_solution(
    space_graph,
    x_solution,
    box_data=None,
    start_idx=None,
    end_idx=None,
    show_arrows=True,
    arrow_ratio=0.18,        # tỷ lệ chiều dài mũi tên trên chiều dài edge
    arrow_sizeref=0.25       # kích thước cone
):
    fig = go.Figure()

    level_colors = {
        0: "royalblue",
        1: "orange",
        2: "green",
        3: "red"
    }

    selected_edges = {k for k, val in zip(space_graph.keys(), x_solution) if val > 0.5}

    all_points = []

    # 1) Draw cubic wireframe background
    if box_data is not None and len(box_data) > 0:
        xs_wf, ys_wf, zs_wf = build_wireframe_arrays(
            box_data,
            scale=voxel_size_m,
            swap_yz=False
        )
        fig.add_trace(go.Scatter3d(
            x=xs_wf, y=ys_wf, z=zs_wf,
            mode="lines",
            line=dict(color="deepskyblue", width=1.2),
            opacity=0.30,
            name=f"Cubic wireframe ({len(box_data)})"
        ))

        for b in box_data:
            pos_m = np.asarray(b["pos"], dtype=float) * np.asarray(voxel_size_m, dtype=float)
            size_m = np.asarray(b["size"], dtype=float) * np.asarray(voxel_size_m, dtype=float)
            all_points.append(pos_m)
            all_points.append(pos_m + size_m)

    # 2) Draw edges by v_level (selected bold, unselected faint) + arrows
    for lv in sorted({d["v_level"] for d in space_graph.values()}):
        xs_sel, ys_sel, zs_sel = [], [], []
        xs_unsel, ys_unsel, zs_unsel = [], [], []

        # arrow buffers
        ax_sel, ay_sel, az_sel, au_sel, av_sel, aw_sel = [], [], [], [], [], []
        ax_uns, ay_uns, az_uns, au_uns, av_uns, aw_uns = [], [], [], [], [], []

        for k, data in space_graph.items():
            if data["v_level"] != lv:
                continue

            x1, y1, z1 = data["startpoint_m"]
            x2, y2, z2 = data["endpoint_m"]

            dx, dy, dz = (x2 - x1), (y2 - y1), (z2 - z1)

            if k in selected_edges:
                xs_sel += [x1, x2, None]
                ys_sel += [y1, y2, None]
                zs_sel += [z1, z2, None]

                if show_arrows:
                    tx = x1 + (1 - arrow_ratio) * dx
                    ty = y1 + (1 - arrow_ratio) * dy
                    tz = z1 + (1 - arrow_ratio) * dz
                    ax_sel.append(tx); ay_sel.append(ty); az_sel.append(tz)
                    au_sel.append(arrow_ratio * dx); av_sel.append(arrow_ratio * dy); aw_sel.append(arrow_ratio * dz)
            else:
                xs_unsel += [x1, x2, None]
                ys_unsel += [y1, y2, None]
                zs_unsel += [z1, z2, None]

                if show_arrows:
                    tx = x1 + (1 - arrow_ratio) * dx
                    ty = y1 + (1 - arrow_ratio) * dy
                    tz = z1 + (1 - arrow_ratio) * dz
                    ax_uns.append(tx); ay_uns.append(ty); az_uns.append(tz)
                    au_uns.append(arrow_ratio * dx); av_uns.append(arrow_ratio * dy); aw_uns.append(arrow_ratio * dz)

            all_points.append([x1, y1, z1])
            all_points.append([x2, y2, z2])

        color = level_colors.get(lv, "black")

        fig.add_trace(go.Scatter3d(
            x=xs_unsel, y=ys_unsel, z=zs_unsel,
            mode="lines",
            line=dict(width=2, color=color),
            opacity=0.10,
            showlegend=False
        ))

        fig.add_trace(go.Scatter3d(
            x=xs_sel, y=ys_sel, z=zs_sel,
            mode="lines",
            line=dict(width=6, color=color),
            name=f"v_level={lv}"
        ))

        if show_arrows and len(ax_uns) > 0:
            fig.add_trace(go.Cone(
                x=ax_uns, y=ay_uns, z=az_uns,
                u=au_uns, v=av_uns, w=aw_uns,
                anchor="tail",
                sizemode="scaled",
                sizeref=arrow_sizeref,
                colorscale=[[0, color], [1, color]],
                showscale=False,
                opacity=0.20,
                showlegend=False,
                hoverinfo="skip"
            ))

        if show_arrows and len(ax_sel) > 0:
            fig.add_trace(go.Cone(
                x=ax_sel, y=ay_sel, z=az_sel,
                u=au_sel, v=av_sel, w=aw_sel,
                anchor="tail",
                sizemode="scaled",
                sizeref=arrow_sizeref,
                colorscale=[[0, color], [1, color]],
                showscale=False,
                opacity=0.90,
                showlegend=False,
                hoverinfo="skip"
            ))

    # 3) Highlight start/end cubic
    if box_data is not None and start_idx is not None and 0 <= int(start_idx) < len(box_data):
        xs_s, ys_s, zs_s = build_wireframe_arrays(
            [box_data[int(start_idx)]],
            scale=voxel_size_m,
            swap_yz=False
        )
        fig.add_trace(go.Scatter3d(
            x=xs_s, y=ys_s, z=zs_s,
            mode="lines",
            line=dict(color="lime", width=6),
            name=f"START cubic (idx={int(start_idx)})"
        ))

    if box_data is not None and end_idx is not None and 0 <= int(end_idx) < len(box_data):
        xs_e, ys_e, zs_e = build_wireframe_arrays(
            [box_data[int(end_idx)]],
            scale=voxel_size_m,
            swap_yz=False
        )
        fig.add_trace(go.Scatter3d(
            x=xs_e, y=ys_e, z=zs_e,
            mode="lines",
            line=dict(color="red", width=6),
            name=f"END cubic (idx={int(end_idx)})"
        ))

    # 4) Axis scale
    if len(all_points) == 0:
        all_points = np.array([[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]])
    else:
        all_points = np.array(all_points, dtype=float)

    xmin, ymin, zmin = all_points.min(axis=0)
    xmax, ymax, zmax = all_points.max(axis=0)

    fig.update_layout(
        title="3D Space Graph with Solution + Cubic Wireframe",
        scene=dict(
            xaxis=dict(title="X (m)", range=[xmin, xmax]),
            yaxis=dict(title="Y (m)", range=[ymin, ymax]),
            zaxis=dict(title="Z (m)", range=[zmin, zmax]),
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

In [ ]:
plot_space_graph_with_solution(
    space_graph,
    x.X,
    box_data=source_boxes_free,
    start_idx=start_box_idx,
    end_idx=end_box_idx
)

In [ ]:
# Tính lại năng lượng của tuyến đường đã chọn từ nghiệm x.X

# 1) Lấy danh sách edge được chọn
selected_edge_keys = [k for k, val in zip(edge_keys, x.X) if val > 0.5]
selected_edges = {k: space_graph[k] for k in selected_edge_keys if k in space_graph}

print(f"So edge duoc chon: {len(selected_edges)}")

# 2) Dựng map start_box -> edge để truy vết đường đi từ start_box_idx
out_map = {}
for k, e in selected_edges.items():
    out_map.setdefault(e["startbox_idx"], []).append(k)

# 3) Truy vết path theo thứ tự từ start -> end
ordered_keys = []
visited = set()
curr_box = start_box_idx

while curr_box is not None and curr_box != end_box_idx:
    candidates = out_map.get(curr_box, [])
    if len(candidates) == 0:
        print(f"Khong tim thay edge di ra tu box {curr_box}.")
        break

    # Nếu có nhiều nhánh, ưu tiên edge có endbox chưa đi qua
    next_key = None
    for ck in candidates:
        if ck not in visited:
            next_key = ck
            break
    if next_key is None:
        print("Phat hien vong lap/khong truy vet tiep duoc.")
        break

    ordered_keys.append(next_key)
    visited.add(next_key)
    curr_box = space_graph[next_key]["endbox_idx"]

if curr_box == end_box_idx:
    print("Da truy vet den box dich.")
else:
    print("Chua den box dich.")

print(f"So edge trong tuyen truy vet: {len(ordered_keys)}")

# 4) Tính lại năng lượng tuyến:
#    - Cong Energy cua tung edge
#    - Cong Etakeoff cua edge dau (neu co)
#    - Cong Elanding cua edge cuoi (neu co)
#    - Cong nang luong chuyen huong giua 2 edge lien tiep
edge_energy_sum = 0.0
transition_energy_sum = 0.0
takeoff_energy = 0.0
landing_energy = 0.0

for i, k in enumerate(ordered_keys):
    e = space_graph[k]
    edge_energy_sum += e.get("Energy", 0.0)

    if i == 0:
        takeoff_energy += e.get("Etakeoff", 0.0)
    if i == len(ordered_keys) - 1:
        landing_energy += e.get("Elanding", 0.0)

for i in range(len(ordered_keys) - 1):
    kin = ordered_keys[i]
    kout = ordered_keys[i + 1]

    ein = space_graph[kin]
    eout = space_graph[kout]

    V_i = np.prod(source_boxes_free[ein["startbox_idx"]]["size"])
    V_j = np.prod(source_boxes_free[eout["endbox_idx"]]["size"])
    w_i = (
        V_i * np.array(source_boxes_free[ein["startbox_idx"]]["avg_wind"]) +
        V_j * np.array(source_boxes_free[eout["endbox_idx"]]["avg_wind"])
    ) / (V_i + V_j)

    trans = calculate_energy_transition(ein, eout, w_i, drone_params, dt)
    transition_energy_sum += trans["total_power_Pi"] * dt

total_energy_recomputed = edge_energy_sum + transition_energy_sum + takeoff_energy + landing_energy

print("\n=== KET QUA TINH LAI NANG LUONG ===")
print(f"Edge Energy tong:        {edge_energy_sum:.2f} J")
print(f"Transition Energy tong:  {transition_energy_sum:.2f} J")
print(f"Takeoff Energy:          {takeoff_energy:.2f} J")
print(f"Landing Energy:          {landing_energy:.2f} J")
print(f"--> Tong Energy tuyen:   {total_energy_recomputed:.2f} J")
print(f"--> Tong Energy tuyen:   {total_energy_recomputed/1000:.4f} kJ")

print("\nThu tu edge trong tuyen:")
for i, k in enumerate(ordered_keys, 1):
    e = space_graph[k]
    print(f"{i:02d}. {k}: {e['startbox_idx']} -> {e['endbox_idx']} | E={e.get('Energy', 0.0):.2f} J")

In [ ]:
from pathlib import Path
import ipynbname

# Kiểm tra/tạo file resultr.csv và lưu kết quả: tên notebook - năng lượng - số cạnh được chọn

result_csv = "resultr.csv"

def get_notebook_name():
    # VS Code Jupyter thường có biến này
    if "__vsc_ipynb_file__" in globals():
        return Path(globals()["__vsc_ipynb_file__"]).name

    # Fallback: ipynbname (nếu có sẵn trong môi trường)
    try:
        return f"{ipynbname.name()}.ipynb"
    except Exception:
        return "unknown_notebook.ipynb"

ten_file = get_notebook_name()
nang_luong = float(total_energy_recomputed) if "total_energy_recomputed" in globals() else np.nan

if "x" in globals() and hasattr(x, "X"):
    so_canh_duoc_chon = int(np.sum(np.array(x.X) > 0.5))
elif "selected_edge_keys" in globals():
    so_canh_duoc_chon = int(len(selected_edge_keys))
else:
    so_canh_duoc_chon = np.nan

# Mở nếu đã có, nếu chưa có thì tạo mới
if Path(result_csv).exists():
    df_result = pd.read_csv(result_csv)
else:
    df_result = pd.DataFrame(columns=["ten_file", "nang_luong", "so_canh_duoc_chon"])

# Upsert theo ten_file: có thì ghi đè, chưa có thì thêm mới
new_row = {
    "ten_file": ten_file,
    "nang_luong": nang_luong,
    "so_canh_duoc_chon": so_canh_duoc_chon
}

mask = df_result["ten_file"].astype(str) == str(ten_file)
if mask.any():
    df_result.loc[mask, ["nang_luong", "so_canh_duoc_chon"]] = [nang_luong, so_canh_duoc_chon]
else:
    df_result.loc[len(df_result)] = new_row

df_result.to_csv(result_csv, index=False)
print(f"Đã cập nhật {result_csv} cho file: {ten_file}")
display(df_result.tail())
